# Custom Keypoint Training

Train a custom keypoint model on your Roboflow dataset (COCO keypoints format).

**Important:** Set **Runtime > Change runtime type > GPU** before starting.

**After running the install cell, RESTART the runtime** (Runtime > Restart session), then run all cells from the top.

## 1. Download Dataset from Roboflow

Export your Roboflow project using the **Keypoint Detection** annotation type.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("your-workspace").project("your-project")
version = project.version(1)
dataset = version.download("coco")

## 2. Install Dependencies

**After this cell finishes, RESTART the runtime (Runtime > Restart session) then run all cells from the top again.**

In [ ]:
# ── Current Colab (Python 3.13+): stock torch/numpy, NOTHING downgraded ─────
# mmcv is the only compiled piece. Dorna hosts a prebuilt wheel in this repo
# (colab_wheels/, built by training_notebooks/build_mmcv_wheel.ipynb).
# When Colab bumps its runtime, rebuild the wheel there and replace it.
import importlib, importlib.util, os, sys, urllib.request

PYTAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
WHEEL_URL = ("https://github.com/dorna-robotics/dorna_vision/raw/pro/colab_wheels/"
             f"mmcv-2.2.0-{PYTAG}-{PYTAG}-linux_x86_64.whl")

def _env_ok(loud=True):
    try:
        import torch, mmcv, mmengine, mmdet, mmpose
        import openvino, nncf
        from mmcv.ops import MultiScaleDeformableAttention   # the compiled part
        if loud:
            print("Environment OK:")
            print(f"  torch:    {torch.__version__}")
            print(f"  mmcv:     {mmcv.__version__}")
            print(f"  mmdet:    {mmdet.__version__}")
            print(f"  mmpose:   {mmpose.__version__}")
            print(f"  openvino: {openvino.__version__}")
        return True
    except Exception:
        return False

if not _env_ok(loud=True):
    print("Installing (~3-4 min)...")
    # modern build tooling — old setuptools breaks on Python 3.13
    !pip -q install "setuptools>=79,<80" "wheel>=0.45"
    # mmengine via plain pip — NEVER via mim (mim drags in openxlab, which
    # downgrades setuptools/requests/rich and bricks Python 3.13)
    !pip -q install mmengine

    # Prebuilt mmcv matched to this runtime — seconds, no compile
    try:
        urllib.request.urlopen(urllib.request.Request(WHEEL_URL, method="HEAD"), timeout=15)
    except Exception:
        raise SystemExit(
            f"No prebuilt mmcv wheel for this runtime ({PYTAG}) at:\n  {WHEEL_URL}\n"
            "Colab's runtime changed — run training_notebooks/build_mmcv_wheel.ipynb "
            "once and publish the new wheel to colab_wheels/.")
    !pip -q install {WHEEL_URL}

    !pip -q install mmdet==3.3.0
    !pip -q install mmpose

    # Lift the mm* hardcoded version caps (path-resolved, no python3.N paths)
    def _lift_caps(mod, subs):
        spec = importlib.util.find_spec(mod)
        if not spec:
            return
        p = os.path.join(os.path.dirname(spec.origin), "__init__.py")
        t = open(p).read()
        for a, b in subs:
            t = t.replace(a, b)
        open(p, "w").write(t)
    _lift_caps("mmdet", [("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")])
    _lift_caps("mmpose", [("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'"),
                          ("mmcv_maximum_version = '2.1.0'", "mmcv_maximum_version = '2.3.0'")])

    # OpenVINO + NNCF + ONNX export
    !pip -q install openvino nncf onnx onnxruntime

    importlib.invalidate_caches()
    assert _env_ok(loud=False), "install did not verify — read the log above; do NOT continue"
    print("\n>>> Runtime -> Restart session, then run all cells from the top <<<")


## 3. Training Parameters

Adjust `MODEL_SIZE`, `EPOCHS`, `OPTIMIZE`, and `AUG` as needed.

In [ ]:
import os
import json
import shutil
import pickle
import glob
import numpy as np
import torch
from pathlib import Path
from google.colab import files

# ============================================================
# EDIT THIS SECTION
# ============================================================
MODEL_SIZE = "tiny"         # "tiny" | "small" | "medium" | "large"
EPOCHS = 300
BATCH_SIZE = 32
INPUT_W = 256               # model input width (W x H). Typical: 192 x 256 for vertical objects.
INPUT_H = 256
LR = 5e-4                   # proven LR for custom RTMPose fine-tuning on small datasets

# OPTIMIZE = True   -> produces a smaller, faster model (recommended for edge deployment)
#                      Falls back automatically if the optimization is unstable.
# OPTIMIZE = False  -> skip optimization, keep the standard model.
OPTIMIZE = True

# ============================================================
# AUGMENTATION — training-time only, no inference-speed impact.
#   Keep it mild for small datasets: RandomBBoxTransform handles both scale
#   and rotation around the bbox center.
# ============================================================
AUG = {
    "scale_factor":  (0.75, 1.25),   # random zoom range around bbox center
    "rotate_factor": 180,             # max rotation in degrees
    "flip_prob":     0.0,            # enable only when keypoints have a valid left/right swap map
}

project_name = project.name.lower().replace(" ", "_")

print(f"Model size:  {MODEL_SIZE}")
print(f"Epochs:      {EPOCHS}")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Input size:  {INPUT_W} x {INPUT_H}  (W x H)")
print(f"LR:          {LR}")
print(f"Optimize:    {OPTIMIZE}")
print(f"Augmentation keys: {list(AUG.keys())}")

## 4. Inspect Dataset + Extract Keypoint Schema

Read the COCO file to discover keypoint names and skeleton connections; these drive the model head size and the visualization.

In [ ]:
DATA_ROOT = dataset.location
TRAIN_ANN = f"{DATA_ROOT}/train/_annotations.coco.json"
VAL_ANN   = f"{DATA_ROOT}/valid/_annotations.coco.json"
TEST_ANN  = f"{DATA_ROOT}/test/_annotations.coco.json"

with open(TRAIN_ANN) as f:
    train_coco = json.load(f)

# Roboflow injects a phantom workspace-slug category at id 0 that
# may or may not carry a `keypoints` schema field but never has any
# annotations. Pick the first category that has BOTH a keypoints
# schema AND at least one labelled instance — that guarantees we
# train on a real class, not the placeholder.
ann_counts = {}
for ann in train_coco.get("annotations", []):
    ann_counts[ann["category_id"]] = ann_counts.get(ann["category_id"], 0) + 1

kp_category = None
for c in train_coco["categories"]:
    if c.get("keypoints") and ann_counts.get(c["id"], 0) > 0:
        kp_category = c
        break
assert kp_category, "No category with keypoints + annotations found — is the dataset actually a keypoint export?"

KEYPOINT_NAMES = kp_category["keypoints"]
SKELETON = kp_category.get("skeleton", [])
NUM_KEYPOINTS = len(KEYPOINT_NAMES)
CLASS_NAME = kp_category["name"]

print(f"Class:         {CLASS_NAME}")
print(f"Keypoints ({NUM_KEYPOINTS}): {KEYPOINT_NAMES}")
print(f"Skeleton edges: {SKELETON}")

for split, ann_path in [("train", TRAIN_ANN), ("valid", VAL_ANN), ("test", TEST_ANN)]:
    if not os.path.exists(ann_path):
        print(f"  {split}: MISSING")
        continue
    with open(ann_path) as f:
        coco = json.load(f)
    img_count = len(coco.get("images", []))
    ann_count = len(coco.get("annotations", []))
    with_kps = sum(1 for a in coco.get("annotations", []) if a.get("keypoints") and a.get("num_keypoints", 0) > 0)
    print(f"  {split}: {img_count} images, {ann_count} annotations ({with_kps} with keypoints)")

## 5. Build Config & Train

In [ ]:
# Architecture configurations per model size.
# RTMPose = CSPNeXt backbone + RTMCCHead (SimCC 1D heatmaps).
#
# The backbone is loaded via init_cfg (prefix='backbone.') so that only the
# backbone weights are transferred. The head stays fresh so it matches your
# custom number of keypoints — trying to `load_from` the full checkpoint would
# fail silently on the head (shape mismatch → random init) and stall training.
ARCH_CFG = {
    "tiny": {
        "deepen_factor": 0.167,
        "widen_factor":  0.375,
        "in_channels":   384,
        "pretrained":    "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-t_simcc-body7_pt-body7_420e-256x192-026a1439_20230504.pth",
    },
    "small": {
        "deepen_factor": 0.167,
        "widen_factor":  0.5,
        "in_channels":   512,
        "pretrained":    "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-s_simcc-body7_pt-body7_420e-256x192-acd4a1ef_20230504.pth",
    },
    "medium": {
        "deepen_factor": 0.67,
        "widen_factor":  0.75,
        "in_channels":   768,
        "pretrained":    "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.pth",
    },
    "large": {
        "deepen_factor": 1.0,
        "widen_factor":  1.0,
        "in_channels":   1024,
        "pretrained":    "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-l_simcc-body7_pt-body7_420e-256x192-4dba18fc_20230504.pth",
    },
}
assert MODEL_SIZE in ARCH_CFG, f"MODEL_SIZE must be one of {list(ARCH_CFG.keys())}"

SIMCC_SPLIT_RATIO = 3.0  # keep fixed; inference wrapper assumes this


def build_pose_config(cfg_arch, num_keypoints, keypoint_names, skeleton,
                      data_root, train_ann, val_ann,
                      epochs, batch_size, input_w, input_h, lr, aug):
    """Generate training config for custom COCO keypoints data."""
    in_channels = cfg_arch['in_channels']
    eta_min = lr * 0.05
    sigma_x = round(input_w / 192 * 4.9, 2)
    sigma_y = round(input_h / 256 * 5.66, 2)
    half_epoch = epochs // 2

    # mmpose metainfo — auto-generated from COCO category
    kp_info_items = []
    for i, name in enumerate(keypoint_names):
        kp_info_items.append(f"{i}: dict(name='{name}', id={i}, color=[255, 128, 0], type='', swap='')")
    kp_info_str = "{" + ", ".join(kp_info_items) + "}"

    sk_items = []
    for i, (a, b) in enumerate(skeleton or []):
        a_name = keypoint_names[a - 1]
        b_name = keypoint_names[b - 1]
        sk_items.append(f"{i}: dict(link=('{a_name}', '{b_name}'), id={i}, color=[0, 255, 128])")
    sk_info_str = "{" + ", ".join(sk_items) + "}"

    joint_weights = "[" + ", ".join("1.0" for _ in keypoint_names) + "]"
    sigmas = "[" + ", ".join("0.025" for _ in keypoint_names) + "]"

    metainfo_block = f"""dict(
    dataset_name='custom',
    paper_info=dict(),
    keypoint_info={kp_info_str},
    skeleton_info={sk_info_str},
    joint_weights={joint_weights},
    sigmas={sigmas},
)"""

    config = f"""default_scope = 'mmpose'

default_hooks = dict(
    timer=dict(type='IterTimerHook'),
    logger=dict(type='LoggerHook', interval=50),
    param_scheduler=dict(type='ParamSchedulerHook'),
    checkpoint=dict(type='CheckpointHook', interval=10, save_best='coco/AP', rule='greater', max_keep_ckpts=3),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    visualization=dict(type='PoseVisualizationHook', enable=False),
)

custom_hooks = [
    dict(type='EMAHook', ema_type='ExpMomentumEMA', momentum=0.0002, update_buffers=True, priority=49),
]

env_cfg = dict(
    cudnn_benchmark=False,
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
    dist_cfg=dict(backend='nccl'),
)

vis_backends = [dict(type='LocalVisBackend')]
visualizer = dict(type='PoseLocalVisualizer', vis_backends=vis_backends, name='visualizer')
log_processor = dict(type='LogProcessor', window_size=50, by_epoch=True, num_digits=6)
log_level = 'INFO'
load_from = None
resume = False

codec = dict(
    type='SimCCLabel',
    input_size=({input_w}, {input_h}),
    sigma=({sigma_x}, {sigma_y}),
    simcc_split_ratio={SIMCC_SPLIT_RATIO},
    normalize=False,
    use_dark=False,
)

# --- Model ---
model = dict(
    type='TopdownPoseEstimator',
    data_preprocessor=dict(
        type='PoseDataPreprocessor',
        mean=[123.675, 116.28, 103.53],
        std=[58.395, 57.12, 57.375],
        bgr_to_rgb=True,
    ),
    backbone=dict(
        _scope_='mmdet',
        type='CSPNeXt',
        arch='P5',
        expand_ratio=0.5,
        deepen_factor={cfg_arch['deepen_factor']},
        widen_factor={cfg_arch['widen_factor']},
        out_indices=(4,),
        channel_attention=True,
        norm_cfg=dict(type='SyncBN'),
        act_cfg=dict(type='SiLU'),
        init_cfg=dict(
            type='Pretrained',
            prefix='backbone.',
            checkpoint='{cfg_arch['pretrained']}',
        ),
    ),
    head=dict(
        type='RTMCCHead',
        in_channels={in_channels},
        out_channels={num_keypoints},
        input_size=codec['input_size'],
        in_featuremap_size=({input_w // 32}, {input_h // 32}),
        simcc_split_ratio=codec['simcc_split_ratio'],
        final_layer_kernel_size=7,
        gau_cfg=dict(
            hidden_dims=256,
            s=128,
            expansion_factor=2,
            dropout_rate=0.0,
            drop_path=0.0,
            act_fn='SiLU',
            use_rel_bias=False,
            pos_enc=False,
        ),
        loss=dict(type='KLDiscretLoss', use_target_weight=True, beta=10.0, label_softmax=True),
        decoder=codec,
    ),
    test_cfg=dict(flip_test=False),
)

# --- Dataset ---
dataset_type = 'CocoDataset'
data_mode = 'topdown'
data_root = '{data_root}/'
metainfo = {metainfo_block}

train_pipeline = [
    dict(type='LoadImage'),
    dict(type='GetBBoxCenterScale'),
    dict(type='RandomBBoxTransform', scale_factor={tuple(aug['scale_factor'])}, rotate_factor={aug['rotate_factor']}),
    dict(type='TopdownAffineGray', input_size=codec['input_size']),
    dict(type='GenerateTarget', encoder=codec),
    dict(type='PackPoseInputs'),
]

val_pipeline = [
    dict(type='LoadImage'),
    dict(type='GetBBoxCenterScale'),
    dict(type='TopdownAffineGray', input_size=codec['input_size']),
    dict(type='PackPoseInputs'),
]

train_dataloader = dict(
    batch_size={batch_size},
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    dataset=dict(
        type=dataset_type,
        data_root=data_root,
        metainfo=metainfo,
        data_mode=data_mode,
        ann_file='{train_ann}',
        data_prefix=dict(img='train/'),
        pipeline=train_pipeline,
    ),
)

val_dataloader = dict(
    batch_size={batch_size},
    num_workers=2,
    persistent_workers=True,
    drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False, round_up=False),
    dataset=dict(
        type=dataset_type,
        data_root=data_root,
        metainfo=metainfo,
        data_mode=data_mode,
        ann_file='{val_ann}',
        data_prefix=dict(img='valid/'),
        test_mode=True,
        pipeline=val_pipeline,
    ),
)

test_dataloader = val_dataloader
val_evaluator = dict(type='CocoMetric', ann_file=data_root + '{val_ann}')
test_evaluator = val_evaluator

train_cfg = dict(by_epoch=True, max_epochs={epochs}, val_interval=10)
val_cfg = dict()
test_cfg = dict()

optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=dict(type='AdamW', lr={lr}, weight_decay=0.0),
    paramwise_cfg=dict(norm_decay_mult=0, bias_decay_mult=0, bypass_duplicate=True),
)

param_scheduler = [
    dict(type='LinearLR', begin=0, end=50, start_factor=1e-5, by_epoch=False),
    dict(type='CosineAnnealingLR', eta_min={eta_min}, begin={half_epoch}, end={epochs}, T_max={epochs - half_epoch}, by_epoch=True, convert_to_iter_based=True),
]

auto_scale_lr = dict(base_batch_size=1024)
"""
    return config


# ---------- Patch num_keypoints field if missing from Roboflow export ----------
# mmpose uses this field to weight per-instance losses; missing/None breaks training.
for split in ("train", "valid", "test"):
    p = os.path.join(DATA_ROOT, split, "_annotations.coco.json")
    if not os.path.exists(p):
        continue
    with open(p) as f:
        coco = json.load(f)
    patched = False
    for ann in coco["annotations"]:
        kps = ann.get("keypoints", [])
        correct = sum(1 for i in range(2, len(kps), 3) if kps[i] > 0)
        if ann.get("num_keypoints") != correct:
            ann["num_keypoints"] = correct
            patched = True
    if patched:
        with open(p, "w") as f:
            json.dump(coco, f)
        print(f"  Patched num_keypoints in {split}")


# ---------- build + train ----------
# ------- Custom TopdownAffine with gray (128, 128, 128) border -------
# Replaces mmpose's default black border so training matches ROI letterbox convention.
import cv2 as _cv2
from mmpose.datasets.transforms import TopdownAffine as _TDA
from mmpose.registry import TRANSFORMS as _TDT

@_TDT.register_module(name='TopdownAffineGray', force=True)
class TopdownAffineGray(_TDA):
    def transform(self, results):
        _orig = _cv2.warpAffine
        def _patched(src, M, dsize, dst=None, flags=_cv2.INTER_LINEAR,
                     borderMode=_cv2.BORDER_CONSTANT, borderValue=None):
            if borderValue is None or borderValue == 0 or borderValue == (0, 0, 0):
                borderValue = (128, 128, 128)
            return _orig(src, M, dsize, dst=dst, flags=flags,
                         borderMode=borderMode, borderValue=borderValue)
        _cv2.warpAffine = _patched
        try:
            return super().transform(results)
        finally:
            _cv2.warpAffine = _orig
# ------------------------------------------------------------

from mmengine.config import Config
from mmengine.runner import Runner

os.makedirs("/content/configs", exist_ok=True)
config_path = f"/content/configs/keypoint_{MODEL_SIZE}_{project_name}.py"

train_ann_rel = "train/_annotations.coco.json"
val_ann_rel   = "valid/_annotations.coco.json"

config_text = build_pose_config(
    ARCH_CFG[MODEL_SIZE], NUM_KEYPOINTS, KEYPOINT_NAMES, SKELETON,
    DATA_ROOT, train_ann_rel, val_ann_rel,
    EPOCHS, BATCH_SIZE, INPUT_W, INPUT_H, LR, AUG,
)
with open(config_path, "w") as f:
    f.write(config_text)
print(f"Config: {config_path}")

work_dir = f"/content/work_dirs/keypoint_{MODEL_SIZE}_{project_name}"
cfg = Config.fromfile(config_path)
cfg.work_dir = work_dir
runner = Runner.from_cfg(cfg)
runner.train()
print(f"\nTraining complete. Work dir: {work_dir}")

## 6. Export & Optimize Model

In [ ]:
import cv2
import openvino as ov
import nncf
from mmpose.apis import init_model

# Pin the LEGACY onnx exporter: torch >= 2.6 defaults dynamo=True, and the
# export wrapper below is written against the legacy tracer.
import torch.onnx
if not hasattr(torch.onnx, "_dynamo_patched"):
    _orig_export = torch.onnx.export
    def _patched_export(*args, **kwargs):
        kwargs["dynamo"] = False
        try:
            return _orig_export(*args, **kwargs)
        except TypeError:      # older torch: no dynamo kwarg
            kwargs.pop("dynamo", None)
            return _orig_export(*args, **kwargs)
    torch.onnx.export = _patched_export
    torch.onnx._dynamo_patched = True

# torch >= 2.6 defaults torch.load(weights_only=True); our OWN checkpoints
# carry numpy scalars in their metadata and are trusted — load them fully.
if not hasattr(torch.serialization, "_wo_patched"):
    _orig_load = torch.load
    def _patched_load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _orig_load(*args, **kwargs)
    torch.load = _patched_load
    torch.serialization._wo_patched = True


# ---------- pick best checkpoint ----------
ckpts = sorted(glob.glob(os.path.join(work_dir, "best_*.pth")))
if not ckpts:
    ckpts = sorted(glob.glob(os.path.join(work_dir, "epoch_*.pth")))
assert ckpts, "No checkpoint found!"
checkpoint = ckpts[-1]
print(f"Checkpoint: {checkpoint}")

# Use EMA weights for export (better eval accuracy)
raw_ckpt = torch.load(checkpoint, map_location="cpu")
if "ema_state_dict" in raw_ckpt:
    raw_ckpt["state_dict"] = raw_ckpt["ema_state_dict"]
    print("Using averaged weights")
fixed_ckpt = os.path.join(work_dir, "best_for_export.pth")
torch.save(raw_ckpt, fixed_ckpt)

# ---------- Load model ----------
pose_model = init_model(config_path, fixed_ckpt, device="cpu")
pose_model.eval()


# ---------- Export wrapper: keypoints + per-keypoint confidence ----------
# Output shape: (batch, num_keypoints, 3) = [x_in_input, y_in_input, confidence]
# Model takes a cropped+resized input of shape (1, 3, INPUT_H, INPUT_W).

class PoseExportWrapper(torch.nn.Module):
    def __init__(self, model, simcc_split_ratio=SIMCC_SPLIT_RATIO):
        super().__init__()
        self.model = model
        self.simcc_split_ratio = float(simcc_split_ratio)

    def forward(self, x):
        feats = self.model.backbone(x)
        simcc_x, simcc_y = self.model.head(feats)
        # simcc_x: (B, K, Wx)  -- Wx = INPUT_W * simcc_split_ratio
        # simcc_y: (B, K, Wy)  -- Wy = INPUT_H * simcc_split_ratio
        conf_x, loc_x = simcc_x.max(dim=-1)
        conf_y, loc_y = simcc_y.max(dim=-1)
        coord_x = loc_x.float() / self.simcc_split_ratio
        coord_y = loc_y.float() / self.simcc_split_ratio
        conf = torch.minimum(conf_x, conf_y)
        return torch.stack([coord_x, coord_y, conf], dim=-1)


wrapped = PoseExportWrapper(pose_model)
wrapped.eval()

os.makedirs("/content/export", exist_ok=True)
onnx_path = "/content/export/model.onnx"
dummy = torch.randn(1, 3, INPUT_H, INPUT_W)

torch.onnx.export(
    wrapped, dummy, onnx_path,
    opset_version=17,
    input_names=["input"],
    output_names=["keypoints"],
)
print(f"Exported model: {onnx_path} ({os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB)")

# ---------- Standard-precision export ----------
core = ov.Core()
ov_model = core.read_model(onnx_path)
std_xml = "/content/export/model.xml"
ov.save_model(ov_model, std_xml)
std_bin = std_xml.replace(".xml", ".bin")
print(f"Standard model: {std_bin} ({os.path.getsize(std_bin) / 1024 / 1024:.1f} MB)")

# ---------- Calibration: cropped object patches resized to (INPUT_W, INPUT_H) ----------
MEAN = np.array([123.675, 116.28, 103.53], dtype=np.float32)
STD  = np.array([58.395, 57.12, 57.375], dtype=np.float32)

def load_object_crops(ann_path, img_dir, input_w, input_h, limit=200, pad=0.15):
    """Extract object crops from the dataset for quantization calibration."""
    with open(ann_path) as f:
        coco = json.load(f)
    id_to_img = {img["id"]: img for img in coco["images"]}
    samples = []
    for ann in coco["annotations"]:
        if len(samples) >= limit:
            break
        if not ann.get("bbox"):
            continue
        img_info = id_to_img.get(ann["image_id"])
        if not img_info:
            continue
        img = cv2.imread(os.path.join(img_dir, img_info["file_name"]))
        if img is None:
            continue
        x, y, w, h = ann["bbox"]
        cx, cy = x + w * 0.5, y + h * 0.5
        side_w = w * (1 + pad)
        side_h = h * (1 + pad)
        x1 = int(max(0, cx - side_w * 0.5))
        y1 = int(max(0, cy - side_h * 0.5))
        x2 = int(min(img.shape[1], cx + side_w * 0.5))
        y2 = int(min(img.shape[0], cy + side_h * 0.5))
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        crop = cv2.resize(crop, (input_w, input_h))
        # mmpose preprocess: BGR -> RGB -> normalize
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32)
        tensor = ((crop - MEAN) / STD).transpose(2, 0, 1)[None]
        samples.append(tensor)
    return samples


opt_xml = None
opt_bin = None

if OPTIMIZE:
    print("\nOptimizing model...")
    calib_tensors = load_object_crops(TRAIN_ANN, f"{DATA_ROOT}/train", INPUT_W, INPUT_H, limit=200)
    print(f"  Calibration samples: {len(calib_tensors)}")
    assert calib_tensors, "No usable crops for calibration"

    ov_model_for_opt = core.read_model(onnx_path)
    calib_dataset = nncf.Dataset(calib_tensors, lambda x: x)
    optimized = nncf.quantize(
        ov_model_for_opt,
        calib_dataset,
        preset=nncf.QuantizationPreset.MIXED,
        subset_size=len(calib_tensors),
    )
    opt_xml_candidate = "/content/export/model_opt.xml"
    ov.save_model(optimized, opt_xml_candidate)
    opt_bin_candidate = opt_xml_candidate.replace(".xml", ".bin")
    print(f"  Optimized model: {opt_bin_candidate} ({os.path.getsize(opt_bin_candidate) / 1024 / 1024:.1f} MB)")
    print(f"  Size reduction: {100 * (1 - os.path.getsize(opt_bin_candidate) / os.path.getsize(std_bin)):.0f}%")

    # Stability: output should be the same keypoint pattern as standard model
    compiled_opt = core.compile_model(optimized, "CPU")
    compiled_std = core.compile_model(ov_model, "CPU")
    probe = calib_tensors[0]
    opt_out = np.asarray(list(compiled_opt([probe]).values())[0])  # (1, K, 3)
    std_out = np.asarray(list(compiled_std([probe]).values())[0])
    # Compare keypoint locations (ignore confidence scale)
    diff = np.mean(np.linalg.norm(opt_out[..., :2] - std_out[..., :2], axis=-1))
    max_side = max(INPUT_W, INPUT_H)
    rel_err = diff / max_side
    print(f"  Stability check: mean keypoint shift = {diff:.2f} px ({rel_err*100:.2f}% of input size)")

    if rel_err < 0.05:  # <5% drift
        opt_xml = opt_xml_candidate
        opt_bin = opt_bin_candidate
        print("  Optimization OK — packaging optimized version")
    else:
        print("  Optimization unstable — packaging standard version only")
else:
    print(f"\nOPTIMIZE={OPTIMIZE} — packaging standard version only.")

## 7. Save & Download Pickle

In [ ]:
# Pickle format mirrors OD/ROD/ISEG:
#   { bin, xml, cls, colors, meta }
# meta.type is 'kp' so the inference side can dispatch to the keypoint path.
# meta also stores keypoint names, skeleton, and input size so inference can
# preprocess + draw correctly without needing the Roboflow project.

if opt_xml and opt_bin and os.path.exists(opt_xml):
    final_xml_path = opt_xml
    final_bin_path = opt_bin
    precision_tag = "int8"
    variant = "optimized"
else:
    final_xml_path = std_xml
    final_bin_path = std_bin
    precision_tag = "fp32"
    variant = "standard"

with open(final_xml_path, "r", encoding="utf-8") as f:
    xml_data = f.read()
with open(final_bin_path, "rb") as f:
    bin_data = f.read()

model_dict = {
    "bin": bin_data,
    "xml": xml_data,
    "cls": [CLASS_NAME],
    "colors": project.colors,
    "keypoint_names": KEYPOINT_NAMES,
    "skeleton": SKELETON,
    "meta": {
        "model_type": MODEL_SIZE,
        "type": "kp",
        "input_w": INPUT_W,
        "input_h": INPUT_H,
        "precision": precision_tag,
        "num_keypoints": NUM_KEYPOINTS,
    },
}

pickle_path = f"/content/{project_name}.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(model_dict, f)

print(f"Pickle contains {variant} model ({len(bin_data) / 1024 / 1024:.1f} MB)")
print(f"\nSaved: {pickle_path}")
print(f"Total size: {os.path.getsize(pickle_path) / 1024 / 1024:.1f} MB")
print(f"Model size: {MODEL_SIZE}")
print(f"Class: {CLASS_NAME}")
print(f"Keypoints: {KEYPOINT_NAMES}")
print(f"Input size: {INPUT_W} x {INPUT_H}")

files.download(pickle_path)